# Prophet - Walmart Store Sales Forecasting

Person B track, fourth and last of four deliverables (DLinear, N-BEATS, TFT,
Prophet). Structurally different from the three darts models: **Prophet has
no global/panel training mode** - it always fits one model per series, so
this notebook cannot reuse the DLinear/N-BEATS/TFT harness directly.

Structure:
- Theory section on Prophet's additive decomposition and why it needs a
  different scope decision than the global darts models (see below).
- A custom Walmart-holidays dataframe for Prophet's `holidays=` parameter,
  including a synthetic "week 51" Christmas-peak event (the dataset marks
  the *following* week as the holiday - see CLAUDE.md / EDA findings).
- Feature-selection runs comparing a holidays-only baseline against adding
  `days_to_christmas`/`days_to_thanksgiving` regressors.
- Diagnostics on the company-level aggregate (with Prophet's own
  trend/seasonality/holiday component plot) and the same 3 representative
  high-volume `(Store, Dept)` series the ARIMA notebook uses, for direct
  comparability.
- 3-fold expanding-window CV on the aggregate series (same shared splits as
  every other model), one MLflow run per fold.
- A full-coverage Kaggle submission via `src.prophet_forecaster`: one
  Prophet model per **Store** (45 series, fit in parallel) with **dept-level
  disaggregation** by historical sales share - the scope decision that keeps
  this notebook inside its ~30 minute time budget instead of fitting
  ~3,000 per-(Store, Dept) Prophet models.

Reuses `src/preprocessing.py` (`load_raw_data`), `src/features.py`
(`add_temporal_features`, `WalmartFeatureEngineer`, `HOLIDAY_DATES`),
`src/validation.py` (shared CV splitter), and `src/evaluation.py`
(`weighted_mae`). Logs to the shared DagsHub MLflow server under the
`Prophet_Training` experiment.

##  Init cell (Colab-compatible)

Byte-identical to the DLinear/N-BEATS/TFT notebooks' init cell (Person B's
track), minus the `darts[torch]` install those need and this notebook
doesn't - `prophet` is already covered by `requirements.txt`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IS_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ

if IS_COLAB:
    REPO_URL = "https://github.com/NikaMikeltadze/walmart-sales-forecasting.git"
    REPO_DIR = "/content/walmart-sales-forecasting"

    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r",
         f"{REPO_DIR}/requirements.txt", "--quiet"],
        check=True,
    )

    os.chdir(f"{REPO_DIR}/notebooks")

    from google.colab import drive
    drive.mount("/content/drive")

    drive_data_dir = Path("/content/drive/MyDrive/walmart-sales-forecasting/data/raw")
    repo_data_dir = Path(REPO_DIR) / "data" / "raw"
    if drive_data_dir.exists():
        subprocess.run(["cp", "-r", f"{drive_data_dir}/.", str(repo_data_dir)], check=True)
    else:
        raise FileNotFoundError(
            f"Expected Drive data folder not found at {drive_data_dir}. "
            "Create it (or add it as a My Drive shortcut) before running this notebook."
        )

sys.path.append(str(Path.cwd().parent))

##  Imports

In [ ]:
import logging
import warnings

import joblib
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from prophet import Prophet

from src.evaluation import weighted_mae
from src.features import HOLIDAY_DATES, WalmartFeatureEngineer, add_temporal_features
from src.preprocessing import load_raw_data
from src.prophet_forecaster import ProphetForecastPipeline, fit_store_models
from src.validation import describe_split, expanding_window_splits, split_frame

warnings.filterwarnings("ignore")  # pmdarima-style convergence/deprecation chatter
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)  # silence per-chain fit spam
logging.getLogger("prophet").setLevel(logging.WARNING)
pd.set_option("display.max_columns", None)

DATA_DIR = Path.cwd().parent / "data" / "raw"
SUBMISSIONS_DIR = Path.cwd().parent / "submissions"
SUBMISSIONS_DIR.mkdir(exist_ok=True)

###  Manual credentials override (VS Code / non-Colab-UI runtimes)

`google.colab.userdata` (Colab Secrets) can only be read from the Colab
**browser UI**. When the Colab runtime is driven from VS Code or any other
non-UI frontend it times out (`Secrets can only be fetched when running from
the Colab UI`). This cell sets the DagsHub creds directly instead, and the
credentials cell below skips `userdata` whenever these env vars are already set.

`getpass` is used so the token is never written into this committed notebook -
run the cell and paste the values when prompted. Leave a prompt blank to fall
through to Colab Secrets / `.env` below (e.g. when you *are* on the Colab UI).

In [ ]:
import os
from getpass import getpass

# Only prompt for values not already set, so re-running cells doesn't re-ask.
# Leave a prompt blank to fall through to Colab Secrets / .env in the next cell.
if not os.environ.get("MLFLOW_TRACKING_USERNAME"):
    _user = getpass("DagsHub username (blank -> use Colab Secrets / .env): ").strip()
    if _user:
        os.environ["MLFLOW_TRACKING_USERNAME"] = _user
if not os.environ.get("MLFLOW_TRACKING_PASSWORD"):
    _token = getpass("DagsHub token (blank -> use Colab Secrets / .env): ").strip()
    if _token:
        os.environ["MLFLOW_TRACKING_PASSWORD"] = _token

##  Load DagsHub credentials

`MLFLOW_TRACKING_USERNAME`/`MLFLOW_TRACKING_PASSWORD` are never hardcoded in
this notebook (it gets committed to the shared repo, so a hardcoded token
would leak).

- On the Colab UI: read from Colab secrets - add `DAGSHUB_USERNAME` and
  `DAGSHUB_TOKEN` via the key icon in the left sidebar, and enable
  "Notebook access" for both. Same approach as every other notebook.
- From VS Code / non-UI runtimes: use the manual-override cell above.
- Locally: falls back to a gitignored `.env` in the repo root.

In [ ]:
if os.environ.get("MLFLOW_TRACKING_USERNAME") and os.environ.get("MLFLOW_TRACKING_PASSWORD"):
    # Already provided (e.g. by the manual-override cell above when driving the
    # Colab runtime from VS Code, where google.colab.userdata would time out).
    # Note: userdata.get(...) must NOT be evaluated in this case - it blocks for
    # ~minutes and raises when there is no Colab browser UI to answer it.
    creds_source = "pre-set environment variables"
elif IS_COLAB:
    from google.colab import userdata

    os.environ["MLFLOW_TRACKING_USERNAME"] = userdata.get("DAGSHUB_USERNAME")
    os.environ["MLFLOW_TRACKING_PASSWORD"] = userdata.get("DAGSHUB_TOKEN")
    creds_source = "Colab secrets (DAGSHUB_USERNAME / DAGSHUB_TOKEN)"
else:
    from dotenv import load_dotenv

    env_path = Path.cwd().parent / ".env"
    load_dotenv(env_path)
    creds_source = str(env_path)

assert os.environ.get("MLFLOW_TRACKING_USERNAME") and os.environ.get("MLFLOW_TRACKING_PASSWORD"), (
    f"MLFLOW_TRACKING_USERNAME/PASSWORD not set (tried: {creds_source}). "
    "On the Colab UI: add DAGSHUB_USERNAME and DAGSHUB_TOKEN as Colab secrets "
    "(key icon in the left sidebar) and enable notebook access for both. "
    "From VS Code / non-UI runtimes: run the manual-override cell above. "
    "Locally: create a .env in the repo root with MLFLOW_TRACKING_USERNAME=... "
    "and MLFLOW_TRACKING_PASSWORD=..."
)
print("MLflow credentials loaded from:", creds_source)

##  MLflow tracking store

Shared DagsHub MLflow server - the single source of truth for cross-model
WMAE comparison and the Model Registry, so all 7 models (both tracks) log
here rather than to a per-notebook local store. Does not silently fall back
to a local store if auth fails - that would desync Prophet's runs from the
rest of the team's.

In [ ]:
MLFLOW_TRACKING_URI = "https://dagshub.com/NikaMikeltadze/walmart-sales-forecasting.mlflow"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

try:
    mlflow.set_experiment("Prophet_Training")
    mlflow.MlflowClient().search_experiments(max_results=1)  # force a network round trip now
except Exception as e:
    raise RuntimeError(
        "Could not authenticate to the DagsHub MLflow server at "
        f"{MLFLOW_TRACKING_URI}. Set MLFLOW_TRACKING_USERNAME and "
        "MLFLOW_TRACKING_PASSWORD (a DagsHub access token) as environment "
        "variables, then re-run this cell. Not falling back to local "
        "./mlruns or sqlite - that would desync from Person A's XGBoost/"
        "LightGBM runs and the rest of the team's runs."
    ) from e

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Active experiment:", mlflow.get_experiment_by_name("Prophet_Training").name)

## Load raw data

In [ ]:
raw = load_raw_data(DATA_DIR)
for name, df in raw.items():
    print(f"{name}: {df.shape}")

assert raw["train"].shape[0] == 421_570, "unexpected train row count - check data/raw"
assert raw["test"].shape[0] > 0
assert raw["stores"].shape[0] == 45

## Prophet - theory and scope decision

**Prophet** (Meta/Facebook, Taylor & Letham 2017) is an additive
decomposable time-series model:

`y(t) = trend(t) + seasonality(t) + holidays(t) + regressors(t) + error(t)`

- **trend(t):** a piecewise-linear (or logistic-growth) curve with
  automatically detected changepoints, so the growth rate itself can shift
  over time - unlike ARIMA's differencing-to-stationarity approach, Prophet
  models the trend directly and explicitly.
- **seasonality(t):** partial Fourier sums fit per seasonal period (yearly,
  weekly, daily). Only **yearly** seasonality is meaningful here - the data
  is already weekly-aggregated, so there is no day-of-week or intra-day
  pattern for `weekly_seasonality`/`daily_seasonality` to capture, and both
  are disabled below.
- **holidays(t):** a per-date step effect from a user-supplied `holidays`
  dataframe - this is where the project's 4 Walmart holidays (and the
  week-51 Christmas-peak-vs-marked-week mismatch from CLAUDE.md/EDA) get
  encoded explicitly, see the next section.
- **regressors(t):** optional extra linear terms (`add_regressor`) for
  covariates known ahead of time. `days_to_christmas`/`days_to_thanksgiving`
  from `src.features.add_temporal_features` fit this well - they encode
  holiday proximity continuously and are computable for any future date.
- Fit is MAP estimation (`mcmc_samples=0`, Prophet's default) via
  `cmdstanpy` - fast (roughly 1-3 seconds per series here), not full
  Bayesian sampling. This is what keeps a 45-series store-level fit
  tractable inside this notebook's time budget.

**Key structural difference from every other model in this project:**
Prophet has **no global/panel training mode** - it always fits one model per
series. The darts models (DLinear/N-BEATS/TFT) are global models trained
once across all ~3,000 `(Store, Dept)` series; XGBoost/LightGBM are single
models over a shared feature matrix; even ARIMA's production path wraps one
shared strategy class. Prophet cannot do this - "training Prophet" always
means "fit N independent per-series models," and N matters a lot for
runtime.

**Scope decision (per the person_b guide, section 0):** fitting ~3,000
per-`(Store, Dept)` Prophet models would blow this notebook's ~30 minute
budget (even at ~1-2s/fit, ~3,000 sequential fits approaches an hour, and
Stan's per-model startup overhead compounds it). Instead this notebook fits
Prophet at **store level (45 series)** - capturing store-specific
trend/seasonality without the ~3,000-fit cost - and recovers `(Store,
Dept)`-level `Weekly_Sales` by scaling each store's forecast by that
department's historical share of the store's total sales
(`src/prophet_forecaster.py`). This is the guide's own "aggregate level"
option, at store-level granularity rather than a single company-wide
series: 45 independent fits give a materially better fit than one national
trend/seasonality curve, while staying just as fast since all 45 fit in
parallel via `joblib`.

## Build the series to model

- **Company aggregate:** total weekly sales summed across all stores/depts -
  used for the feature-selection, diagnostic, and CV runs below (mirrors the
  ARIMA notebook's theory-focused approach).
- **Representative series:** the same 3 highest-volume `(Store, Dept)`
  series (with full-length history) that the ARIMA notebook uses, so
  Prophet's per-series WMAE is directly comparable across the two classical
  -model notebooks.
- **Store-level series** (45, one per Store): the actual granularity used
  for the production submission strategy - see "Validating the actual
  submission strategy" further down.

A week is treated as a holiday week if any row that week is flagged
`IsHoliday` (WMAE weights holiday weeks 5x) - same convention as ARIMA.

In [ ]:
train = raw["train"].copy()
train["Date"] = pd.to_datetime(train["Date"])

# Company-level weekly total sales (single aggregate series).
agg = train.groupby("Date")["Weekly_Sales"].sum().sort_index()

# Per-week holiday flag, shared by the aggregate scoring below.
holiday_by_date = train.groupby("Date")["IsHoliday"].any()

# 3 representative high-volume (Store, Dept) series with full-length history -
# same selection logic as the ARIMA notebook, for direct comparability.
series_len = train.groupby(["Store", "Dept"])["Date"].nunique()
full_len = series_len.max()
eligible = series_len[series_len == full_len].index
series_means = train.groupby(["Store", "Dept"])["Weekly_Sales"].mean()
rep_keys = series_means.loc[eligible].sort_values(ascending=False).head(3).index.tolist()
rep_keys = [(int(s), int(d)) for s, d in rep_keys]


def build_series(store, dept):
    """Return (values Series, holiday Series) both indexed by Date for one series."""
    g = train[(train["Store"] == store) & (train["Dept"] == dept)].sort_values("Date")
    values = g.set_index("Date")["Weekly_Sales"]
    values = values[~values.index.duplicated(keep="last")]
    holiday = g.groupby("Date")["IsHoliday"].any().reindex(values.index).fillna(False)
    return values, holiday


print("Aggregate series:", agg.shape[0], "weeks,",
      agg.index.min().date(), "->", agg.index.max().date())
print("Representative (Store, Dept) series:", rep_keys)
agg.tail(3)

## Custom Walmart holidays for Prophet

Prophet's `holidays` parameter takes a dataframe of `(ds, holiday)` step
events. The 4 Walmart holiday dates from `src.features.HOLIDAY_DATES` are
themselves the Friday week-ending dates this dataset already uses (the same
convention `IsHoliday` is marked on), so they line up exactly with the
weekly `ds` grid - no date-window guessing needed for Super Bowl, Labor Day,
or Thanksgiving.

**Christmas is the documented exception** (CLAUDE.md / EDA): the dataset
marks the *last* week of the year as the Christmas holiday, but the real
sales peak lands one week earlier (week 51). Rather than smearing a single
holiday effect across both weeks with a window, this adds a **second,
separate holiday event** - `christmas_peak_week51`, dated exactly 7 days
before the marked date - so Prophet can learn two distinct coefficients: the
real pre-Christmas sales spike and the smaller effect of the marked week
itself.

In [ ]:
def build_walmart_holidays():
    rows = []
    for name, year_map in HOLIDAY_DATES.items():
        for year, date_str in year_map.items():
            marked_date = pd.Timestamp(date_str)
            rows.append({"holiday": name, "ds": marked_date, "lower_window": 0, "upper_window": 0})
            if name == "christmas":
                peak_date = marked_date - pd.Timedelta(weeks=1)
                rows.append({
                    "holiday": "christmas_peak_week51", "ds": peak_date,
                    "lower_window": 0, "upper_window": 0,
                })
    return pd.DataFrame(rows)


WALMART_HOLIDAYS = build_walmart_holidays()
print(WALMART_HOLIDAYS["holiday"].value_counts())
WALMART_HOLIDAYS.sort_values("ds").reset_index(drop=True)

## Time-based validation window

Uses the shared `expanding_window_splits` (same splitter as all other
models) so WMAE is comparable. The last fold is used as the reported
holdout.

**Known limitation (same as the tree models and ARIMA):** train ends
2012-11-01, so no validation fold can contain a Thanksgiving or Christmas
week - those only appear in the real `test.csv` horizon. WMAE weights
holiday weeks 5x and those are the two biggest sales holidays, so holdout
WMAE is optimistic relative to the true 39-week horizon.

In [ ]:
splits = expanding_window_splits(pd.Series(agg.index), n_splits=3, val_weeks=13)
print(f"{len(splits)} folds")
for i, split in enumerate(splits):
    print(f"fold {i}:", describe_split(split))

HOLDOUT = splits[-1]
print("\nReported holdout:", describe_split(HOLDOUT))

## MLflow run helper

`log_prophet_run` fits a `Prophet` model on the train portion of a
Date-indexed series, forecasts the holdout window, scores WMAE + MAE, and
logs one flat MLflow run (params = regressors used / CV strategy; metrics =
WMAE, MAE; artifacts = a forecast-vs-actual plot, plus an optional
trend/seasonality/holidays component plot). Same flat-run +
`experiment_group` tag convention the other 6 notebooks use.

In [ ]:
def _slice(series, split):
    train_s = series[series.index <= split.train_end]
    if split.train_start is not None:
        train_s = train_s[train_s.index >= split.train_start]
    val_s = series[(series.index >= split.val_start) & (series.index <= split.val_end)]
    return train_s, val_s


def log_prophet_run(run_name, experiment_group, series, holiday_series, split,
                     holidays_df, regressor_cols=None, extra_params=None,
                     extra_artifacts=None, log_components=False):
    """Fit Prophet on the split's train window, score the val window, log one run."""
    train_s, val_s = _slice(series, split)
    regressor_cols = regressor_cols or []

    regressor_frame = None
    if regressor_cols:
        regressor_frame = add_temporal_features(
            pd.DataFrame({"Date": series.index})
        ).set_index("Date")

    train_df = pd.DataFrame({"ds": train_s.index, "y": train_s.to_numpy()})
    for col in regressor_cols:
        train_df[col] = train_df["ds"].map(regressor_frame[col])

    model = Prophet(
        holidays=holidays_df,
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
    )
    for col in regressor_cols:
        model.add_regressor(col)
    model.fit(train_df)

    future = pd.DataFrame({"ds": val_s.index})
    for col in regressor_cols:
        future[col] = future["ds"].map(regressor_frame[col])
    forecast = model.predict(future)
    preds = forecast["yhat"].to_numpy().clip(min=0.0)

    val_holiday = holiday_series.reindex(val_s.index).fillna(False).to_numpy()
    wmae = weighted_mae(val_s.to_numpy(), preds, val_holiday)
    mae = float(np.mean(np.abs(val_s.to_numpy() - preds)))
    # Same relative-scale reporting as ARIMA's log_arima_run: the aggregate is a
    # much larger absolute number than a single (Store, Dept) series, so WMAE as
    # a % of the series' own mean makes runs comparable at a glance.
    wmae_pct = wmae / float(train_s.mean())

    with mlflow.start_run(run_name=run_name) as run:
        mlflow.set_tag("experiment_group", experiment_group)
        mlflow.log_param("model_family", "Prophet")
        mlflow.log_param("regressors", ",".join(regressor_cols) if regressor_cols else "none")
        mlflow.log_param("n_train_obs", int(len(train_s)))
        mlflow.log_param("series_mean_level", float(train_s.mean()))
        mlflow.log_param("cv_strategy", "expanding_window")
        mlflow.log_params(describe_split(split))
        if extra_params:
            mlflow.log_params(extra_params)

        mlflow.log_metric("wmae", wmae)
        mlflow.log_metric("mae", mae)
        mlflow.log_metric("wmae_pct_of_series_mean", wmae_pct)

        fig, ax = plt.subplots(figsize=(11, 4))
        ax.plot(series.index, series.to_numpy(), color="0.6", label="history")
        ax.plot(val_s.index, val_s.to_numpy(), marker="o", label="actual (val)")
        ax.plot(val_s.index, preds, marker="x", label="forecast")
        ax.axvline(split.train_end, color="k", ls="--", lw=1)
        ax.set_title(f"{run_name}: Prophet WMAE={wmae:,.0f} ({wmae_pct:.1%} of mean level)")
        ax.legend()
        plt.tight_layout()
        plot_path = SUBMISSIONS_DIR / f"_prophet_forecast_{run_name}.png"
        fig.savefig(plot_path)
        plt.show()
        mlflow.log_artifact(str(plot_path))

        if log_components:
            full_frame = pd.concat([train_df, future], ignore_index=True)
            full_forecast = model.predict(full_frame)
            comp_fig = model.plot_components(full_forecast)
            comp_fig.set_size_inches(11, 8)
            plt.tight_layout()
            comp_path = SUBMISSIONS_DIR / f"_prophet_components_{run_name}.png"
            comp_fig.savefig(comp_path, bbox_inches="tight")
            plt.show()
            mlflow.log_artifact(str(comp_path))

        for art in (extra_artifacts or []):
            mlflow.log_artifact(str(art))

        run_id = run.info.run_id

    print(f"{run_name}: WMAE={wmae:,.2f} ({wmae_pct:.2%} of series mean) MAE={mae:,.2f}")
    return {"model": model, "wmae": wmae, "mae": mae, "wmae_pct": wmae_pct, "run_id": run_id}

## Feature selection: holidays-only baseline vs. added regressors

Two small runs on the company aggregate, scored on the shared `HOLDOUT`
fold, decide whether `days_to_christmas`/`days_to_thanksgiving` regressors
earn their keep on top of the custom holidays dataframe. Whichever wins gets
carried forward as `WINNING_REGRESSOR_COLS` into every later run (aggregate,
CV, representative series, and the final store-level submission strategy) -
this is the `{Model}_Feature_Selection` stage CLAUDE.md's MLflow convention
expects.

In [ ]:
base_result = log_prophet_run(
    "Prophet_FeatureSelection_Baseline", "feature_selection", agg, holiday_by_date, HOLDOUT,
    WALMART_HOLIDAYS, regressor_cols=None,
    extra_params={"config": "holidays_only"},
)

CANDIDATE_REGRESSORS = ["days_to_christmas", "days_to_thanksgiving"]
reg_result = log_prophet_run(
    "Prophet_FeatureSelection_Regressors", "feature_selection", agg, holiday_by_date, HOLDOUT,
    WALMART_HOLIDAYS, regressor_cols=CANDIDATE_REGRESSORS,
    extra_params={"config": "holidays_plus_days_to_regressors"},
)

WINNING_REGRESSOR_COLS = CANDIDATE_REGRESSORS if reg_result["wmae"] < base_result["wmae"] else None
print(f"\nBaseline WMAE:   {base_result['wmae']:,.2f}")
print(f"Regressors WMAE: {reg_result['wmae']:,.2f}")
print(f"Winning config: {'regressors' if WINNING_REGRESSOR_COLS else 'baseline (holidays only)'}")

## Experiment: Prophet on the company aggregate

Full diagnostic run on the aggregate series, using the winning
feature-selection config. Also logs Prophet's own `plot_components` figure
(trend / yearly seasonality / holiday effects) - this is the Prophet-native
equivalent of ARIMA's ACF/PACF + seasonal-decomposition plots, and is the
clearest single artifact for checking that the `christmas_peak_week51`
event actually picked up a distinct, sensible coefficient.

**Reading the WMAE:** the aggregate is **summed** company sales (~$47M/week
on average), so its absolute WMAE is naturally far larger than the
per-series WMAE the tree models and the representative-series run below
report - a difference in scale, not model quality. `log_prophet_run` also
logs `wmae_pct_of_series_mean` for exactly this reason.

In [ ]:
agg_result = log_prophet_run(
    "Prophet_Aggregate", "aggregate", agg, holiday_by_date, HOLDOUT,
    WALMART_HOLIDAYS, regressor_cols=WINNING_REGRESSOR_COLS,
    extra_params={"scope": "company_aggregate"},
    log_components=True,
)

## Cross-validation: 3 expanding-window folds

Same 3 folds built above (`splits`), on the aggregate series. Following the
TFT notebook's convention (rather than DLinear/N-BEATS's single-run style),
**each fold gets its own MLflow run** (`Prophet_CV_Fold0/1/2`) so per-fold
diagnostics (forecast plot, params) are individually browsable in the
DagsHub UI. A summary WMAE-by-fold bar chart is generated afterward and
attached to the last fold's run.

In [ ]:
cv_results = []
for i, split in enumerate(splits):
    res = log_prophet_run(
        f"Prophet_CV_Fold{i}", "cv", agg, holiday_by_date, split,
        WALMART_HOLIDAYS, regressor_cols=WINNING_REGRESSOR_COLS,
        extra_params={"fold": i},
    )
    cv_results.append(res)

fold_wmaes = [r["wmae"] for r in cv_results]
print(f"\nCV WMAE by fold: {[round(w, 1) for w in fold_wmaes]}")
print(f"Mean CV WMAE: {np.mean(fold_wmaes):,.2f} +/- {np.std(fold_wmaes):,.2f}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar([f"Fold {i}" for i in range(len(fold_wmaes))], fold_wmaes, color="steelblue")
ax.set_ylabel("WMAE")
ax.set_title("Prophet aggregate-series CV: WMAE by fold")
plt.tight_layout()
CV_BAR_PATH = SUBMISSIONS_DIR / "_prophet_cv_wmae_by_fold.png"
fig.savefig(CV_BAR_PATH)
plt.show()

with mlflow.start_run(run_id=cv_results[-1]["run_id"]):
    mlflow.log_artifact(str(CV_BAR_PATH))

## Experiment: Prophet on representative high-volume series

Prophet on each of the same 3 representative `(Store, Dept)` series ARIMA
uses. These WMAE values are at the **same per-series scale** as the tree
models and ARIMA's `ARIMA_Series_*` runs, but note the same **limitation**
ARIMA states: scoped to only these 3 series (not all ~2,950), illustrative
of Prophet's per-series behavior rather than a full-panel score. The
full-panel number is the store-level + disaggregation strategy validated
below.

In [ ]:
rep_results = []
for store, dept in rep_keys:
    values, holiday = build_series(store, dept)
    res = log_prophet_run(
        f"Prophet_Series_{store}_{dept}", "representative_series",
        values, holiday, HOLDOUT, WALMART_HOLIDAYS,
        regressor_cols=WINNING_REGRESSOR_COLS,
        extra_params={"store": store, "dept": dept, "scope": "single_series"},
    )
    rep_results.append(((store, dept), res))

rep_wmae_mean = float(np.mean([r["wmae"] for _, r in rep_results]))
print(f"\nMean WMAE across {len(rep_results)} representative series: {rep_wmae_mean:,.2f}")

## Validating the actual submission strategy on the shared holdout

The `Prophet_Aggregate` / `Prophet_Series_*` / `Prophet_CV_*` runs above are
theory/diagnostic exhibits on the aggregate or a handful of series - they
are **not** the strategy used for the submission below, and their WMAE
values are not directly comparable to it (different unit scale for the
aggregate, only 3 series for the representative runs).

The strategy actually shipped in the submission is different: **one Prophet
model per Store (45 series, fit in parallel)**, with **(Store, Dept)-level
Weekly_Sales recovered by scaling each store's forecast by that
department's historical share of the store's total sales**
(`src.prophet_forecaster.ProphetForecastPipeline`). This section fits that
exact strategy on the same shared `HOLDOUT` fold's train window and scores
it across **all** `(Store, Dept)` rows in the held-out weeks - giving a WMAE
at the same scale and on the same validation window as the other 6 models'
own holdout/CV numbers, instead of only the diagnostic Prophet numbers
above.

`WalmartFeatureEngineer` (already used by Person A's tree-model track) is
reused here purely for its `store_mean_`/`store_dept_mean_`/`dept_mean_`
aggregates - the dept-level disaggregation shares come directly from those,
rather than recomputing the same groupby logic again.

In [ ]:
train_holdout, val_holdout = split_frame(train, HOLDOUT)

holdout_store_models = fit_store_models(
    train_holdout, WALMART_HOLIDAYS, regressor_cols=WINNING_REGRESSOR_COLS,
    n_jobs=-1, verbose=True,
)

fe_holdout = WalmartFeatureEngineer()
fe_holdout.fit(train_holdout)

holdout_pipeline = ProphetForecastPipeline(
    store_models_json=holdout_store_models,
    store_mean=fe_holdout.store_mean_.to_dict(),
    store_dept_mean={(int(s), int(d)): v for (s, d), v in fe_holdout.store_dept_mean_.items()},
    dept_mean=fe_holdout.dept_mean_.to_dict(),
    global_mean=float(fe_holdout.global_mean_),
    regressor_cols=WINNING_REGRESSOR_COLS,
)

holdout_preds = holdout_pipeline.predict(val_holdout[["Store", "Dept", "Date"]])

final_strategy_wmae = weighted_mae(
    val_holdout["Weekly_Sales"].to_numpy(), holdout_preds, val_holdout["IsHoliday"].to_numpy()
)
final_strategy_mae = float(np.mean(np.abs(val_holdout["Weekly_Sales"].to_numpy() - holdout_preds)))

with mlflow.start_run(run_name="Prophet_Final_HoldoutValidation"):
    mlflow.set_tag("experiment_group", "final_validation")
    mlflow.log_param("strategy", "store_level_prophet + dept_share_disaggregation")
    mlflow.log_param("regressors", ",".join(WINNING_REGRESSOR_COLS) if WINNING_REGRESSOR_COLS else "none")
    mlflow.log_param("n_stores", len(holdout_store_models))
    mlflow.log_params(describe_split(HOLDOUT))
    mlflow.log_metric("wmae", final_strategy_wmae)
    mlflow.log_metric("mae", final_strategy_mae)

print(f"Final submission strategy - holdout WMAE (all Store/Dept rows): {final_strategy_wmae:,.2f}")
print(f"Final submission strategy - holdout MAE: {final_strategy_mae:,.2f}")

## Final fit + full-coverage submission

`src.prophet_forecaster.fit_store_models` + `ProphetForecastPipeline` turn
Prophet into a full 115,064-row submission without fitting a per-series
model for each of the ~2,950 test `(Store, Dept)` combinations. For each
**Store**, one Prophet model (holidays + winning regressor config) is fit on
that store's total weekly sales; every `(Store, Dept)` row is then predicted
by scaling the store-level forecast by that department's historical share
of the store's sales (falling back to the department's average share of
company-wide sales for any combo unseen at fit time).

`fit_store_models` does the actual (parallelized) work; the resulting
`ProphetForecastPipeline` is what gets `joblib.dump`'d and logged to MLflow
- it is the artifact `model_inference.ipynb` will load later and call
`.predict()` on raw `test.csv`-shaped data directly.

In [ ]:
# n_jobs=-1 fans the 45 independent store-level Prophet fits across all CPU
# cores - this is pure CPU work (cmdstanpy's optimizer), no GPU benefit.
final_store_models = fit_store_models(
    raw["train"], WALMART_HOLIDAYS, regressor_cols=WINNING_REGRESSOR_COLS,
    n_jobs=-1, verbose=True,
)

fe_final = WalmartFeatureEngineer()
fe_final.fit(raw["train"])

final_pipeline = ProphetForecastPipeline(
    store_models_json=final_store_models,
    store_mean=fe_final.store_mean_.to_dict(),
    store_dept_mean={(int(s), int(d)): v for (s, d), v in fe_final.store_dept_mean_.items()},
    dept_mean=fe_final.dept_mean_.to_dict(),
    global_mean=float(fe_final.global_mean_),
    regressor_cols=WINNING_REGRESSOR_COLS,
)

MODEL_PATH = SUBMISSIONS_DIR / "_prophet_forecaster.joblib"
joblib.dump(final_pipeline, MODEL_PATH)

with mlflow.start_run(run_name="Prophet_Final") as final_run:
    mlflow.set_tag("experiment_group", "final")
    mlflow.log_param("strategy", "store_level_prophet + dept_share_disaggregation")
    mlflow.log_param("regressors", ",".join(WINNING_REGRESSOR_COLS) if WINNING_REGRESSOR_COLS else "none")
    mlflow.log_param("n_stores", len(final_store_models))
    mlflow.log_param("n_series_dept_level", len(fe_final.store_dept_mean_))

    # Headline metrics: the ACTUAL shipped strategy's own holdout score, computed
    # just above (comparable scale/window to the other models' CV/holdout WMAE).
    mlflow.log_metric("wmae", final_strategy_wmae)
    mlflow.log_metric("mae", final_strategy_mae)

    # Reference-only metrics from the diagnostic Prophet runs - NOT this model's
    # own score (different scope/scale) - kept only for side-by-side context in
    # the MLflow UI, clearly prefixed so they are never mistaken for Prophet_Final's
    # own performance.
    mlflow.log_metric("reference_prophet_aggregate_wmae", agg_result["wmae"])
    mlflow.log_metric("reference_prophet_representative_series_wmae_mean", rep_wmae_mean)

    mlflow.log_artifact(str(MODEL_PATH))
    final_run_id = final_run.info.run_id

print("Logged Prophet_Final, run_id:", final_run_id)
print(f"Prophet_Final headline WMAE (actual shipped strategy, holdout): {final_strategy_wmae:,.2f}")

## Submission

Runs the fitted pipeline directly on raw `test.csv`-shaped data and writes
`submissions/prophet_submission.csv` in the standard `Id` =
`Store_Dept_Date` format, then verifies it matches `sampleSubmission.csv`
row-for-row and attaches it to the `Prophet_Final` run.

In [ ]:
test_preds = final_pipeline.predict(raw["test"])
assert not np.isnan(test_preds).any(), "pipeline produced NaN predictions on raw test data"

submission = pd.DataFrame(
    {
        "Id": (
            raw["test"]["Store"].astype(str)
            + "_"
            + raw["test"]["Dept"].astype(str)
            + "_"
            + raw["test"]["Date"].dt.strftime("%Y-%m-%d")
        ),
        "Weekly_Sales": test_preds,
    }
)

SUBMISSION_PATH = SUBMISSIONS_DIR / "prophet_submission.csv"
submission.to_csv(SUBMISSION_PATH, index=False)
print("Wrote", SUBMISSION_PATH, submission.shape)

# Hard integrity checks (do not depend on any other file).
assert submission.shape == (115064, 2), submission.shape
assert list(submission.columns) == ["Id", "Weekly_Sales"]
assert not submission["Weekly_Sales"].isna().any()

# Optional row-for-row cross-check against sampleSubmission. Wrapped so a locked
# file (e.g. open in Excel) does not break the run - the Id order is already
# guaranteed by construction (same raw test frame the other notebooks used).
try:
    sample = pd.read_csv(DATA_DIR / "sampleSubmission.csv")
    assert submission["Id"].tolist() == sample["Id"].tolist(), "Id order mismatch vs sampleSubmission"
    print("Submission matches sampleSubmission row-for-row:", submission.shape[0], "rows, no NaNs.")
except PermissionError:
    print("Note: sampleSubmission.csv is locked (open elsewhere) - skipped the row-for-row "
          "cross-check. Shape/column/NaN checks passed:", submission.shape)
submission.head()

In [ ]:
with mlflow.start_run(run_id=final_run_id):
    mlflow.log_artifact(str(SUBMISSION_PATH))

print("Submission logged as an artifact on Prophet_Final (run_id:", final_run_id, ")")